In [49]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, BaseMessage
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langchain.agents.structured_output import ProviderStrategy

from langgraph.graph import START, END, StateGraph
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver

from typing import Annotated
from pydantic import BaseModel, Field

from dotenv import load_dotenv

from rich import print

import os
import json

In [50]:
load_dotenv()
openai_key = os.getenv("OPENAI_KEY")

In [51]:
llm = ChatOpenAI(
        model="gpt-4.1-mini",
        temperature=0,
        api_key = openai_key
    )

In [52]:
class FinalOutputSchema(BaseModel):
    result:str = Field(description = "contains the final response returned by the llm")

In [53]:
agent = create_agent(
    model =llm, 
    response_format= ProviderStrategy(FinalOutputSchema)
)

In [54]:
class State(BaseModel):
    messages:Annotated[list[BaseMessage],add_messages]

In [55]:
steps =[
    "step1: Understanding the Problem Statement first and identify what exactly needs to be solved",
    "step2: Breakdown the problems into sub-problems if possible and then go through through each of the subproblem sequentially",
    "Step3: State the final concise answer."
]

In [56]:
system_prompt = """
    You are an expert problem solver.

    You will solve the user's problem step by step.

    For every step:
    - Follow the instruction given for that step.
    - Use the information produced by previous steps.
    - Do not skip the current step.
"""

In [57]:
def ignition(state:State):

    question = """
        A database table has 5 million rows.
        Adding an index makes SELECT queries 10x faster
        but makes INSERT operations 20% slower.

        If the system performs 90% SELECTs and 10% INSERTs,
        what is the overall performance impact?
    """

    user_query = input("Enter your query: ").strip()

    # Use implicit truthiness with .strip() to ignore whitespace-only inputs

    if len(user_query):
        question = user_query

    messages = list(state.messages)
    messages.append(SystemMessage(content=system_prompt))
    messages.append(HumanMessage(content=f"Problem:\n{question}"))

    return {"messages": messages}

In [58]:
def reason_and_respond(state: State):
    messages = list(state.messages)
    new_messages = []

    for step in steps:
        step_message = HumanMessage(content=f"Now perform the following step: {step}")
        messages.append(step_message)
        new_messages.append(step_message)

        result = agent.invoke({"messages":messages})
        ai_message = result["messages"][-1]
        
        messages.append(ai_message)
        new_messages.append(ai_message)

    return {"messages": new_messages}

In [59]:
# initializing the graph
graph = StateGraph(State)

In [60]:
# making the nodes

graph.add_node("ignition", ignition)
graph.add_node("reason_and_respond", reason_and_respond)

In [61]:
# defining the edges

graph.add_edge(START, "ignition")
graph.add_edge("ignition","reason_and_respond")
graph.add_edge("reason_and_respond",END)

In [62]:
checkpointer = InMemorySaver()
workflow = graph.compile(checkpointer=checkpointer) 

In [63]:
config = {
    "configurable": {
        "thread_id": "content-123"
    }
}

In [ ]:
initial_state ={}
result = workflow.invoke(initial_state, config = config)

# print(result)
# print(result["messages"][-1].content)

final_result = result["messages"][-1].content
final_result = json.loads(final_result)["result"]
print(final_result)

The costs of goods are rising primarily due to a combination of factors including increased raw material prices, 
higher labor costs, supply chain disruptions, inflationary pressures, and increased demand. External economic 
conditions such as inflation and global events also contribute to these rising costs, leading to higher prices for 
consumers.

: 